In [2]:
import time
import math
import random
import gmpy2
from typing import List, Tuple, Dict

# =========================================================
# 공통 헬퍼
# =========================================================
def bit_reverse_int(x: int, bits: int) -> int:
    r = 0
    for _ in range(bits):
        r = (r << 1) | (x & 1)
        x >>= 1
    return r

def bit_reverse_permute(vec: List[int]) -> List[int]:
    n = len(vec)
    bits = (n - 1).bit_length()
    out = [0] * n
    for i, v in enumerate(vec):
        out[bit_reverse_int(i, bits)] = v
    return out

# =========================================================
# BFU들 (너의 기존 클래스들을 그대로 사용/추가)
#  - Radix4_BFU, Radix16_BFU는 네 코드 그대로 둔다.
#  - Radix2_BFU / Radix8_BFU는 너의 구현으로 교체하면 됨.
# =========================================================
class Radix4_BFU:
    def __init__(self, mod=97, omega=22):
        self.mod = mod
        self.omega = omega
    def hadamard(self, a, b):
        return (a + b) % self.mod, (a - b) % self.mod
    def process(self, a):
        a = list(a)
        a[0], a[1] = self.hadamard(a[0], a[1])
        a[2], a[3] = self.hadamard(a[2], a[3])
        # 내부 twiddle (네가 쓰던 스케줄)
        a[3] = (a[3] * self.omega) % self.mod
        a[0], a[2] = self.hadamard(a[0], a[2])
        a[1], a[3] = self.hadamard(a[1], a[3])
        return a


import gmpy2

class Radix16_BFU:
    def __init__(self, mod=17, omega=7):
        self.mod = mod
        self.omega = omega  # Twiddle Factor
        self.m = [0] * 18
        self.k = [0] * 5  # (미사용)

        inv_2 = gmpy2.invert(2, self.mod)
        self.m[0]  = gmpy2.mul(gmpy2.add(gmpy2.powmod(self.omega, 3, self.mod),
                                         gmpy2.powmod(self.omega, 5, self.mod)), inv_2) % self.mod
        self.m[1]  = gmpy2.mul(gmpy2.sub(gmpy2.powmod(self.omega, 5, self.mod),
                                         gmpy2.powmod(self.omega, 3, self.mod)), inv_2) % self.mod
        self.m[2]  = gmpy2.mul(gmpy2.add(gmpy2.powmod(self.omega, 1, self.mod),
                                         gmpy2.powmod(self.omega, 7, self.mod)), inv_2) % self.mod
        self.m[3]  = gmpy2.mul(gmpy2.sub(gmpy2.powmod(self.omega, 1, self.mod),
                                         gmpy2.powmod(self.omega, 7, self.mod)), inv_2) % self.mod
        self.m[4]  = gmpy2.add(self.m[2], self.m[0]) % self.mod
        self.m[5]  = gmpy2.add(self.m[1], self.m[3]) % self.mod
        self.m[6]  = self.m[7] = self.m[12] = gmpy2.powmod(self.omega, 4, self.mod)
        self.m[8]  = self.m[10] = gmpy2.mul(gmpy2.add(gmpy2.powmod(self.omega, 2, self.mod),
                                                      gmpy2.powmod(self.omega, 6, self.mod)), inv_2) % self.mod
        self.m[9]  = self.m[11] = gmpy2.mul(gmpy2.sub(gmpy2.powmod(self.omega, 2, self.mod),
                                                      gmpy2.powmod(self.omega, 6, self.mod)), inv_2) % self.mod

    # ✅ 인스턴스 메서드 (self.mod 사용)
    def hadamard(self, a, b):
        out1 = gmpy2.mod(a + b, self.mod)
        out2 = gmpy2.mod(a - b, self.mod)
        return int(out1), int(out2)

    

    def process(self, a):
        s = [0] * 2

        # Stage 1
        for i in range(0, 16, 2):
            a[i], a[i+1] = self.hadamard(a[i], a[i+1])

        # Stage 2
        for i, j in [(0,2), (4,6), (5,7), (8,10), (12,14), (11,13), (9,15)]:
            a[i], a[j] = self.hadamard(a[i], a[j])

        # Stage 3
        s[0] = (a[9]  + a[11]) % self.mod
        s[1] = (a[13] + a[15]) % self.mod
        for i, j in [(10,14), (8,12), (0,4)]:
            a[i], a[j] = self.hadamard(a[i], a[j])

        # Twiddle multiply (원래 자리 덮어쓰기)
        a[9]  = gmpy2.mod(a[9]  * self.m[0],  self.mod)
        a[15] = gmpy2.mod(a[15] * self.m[1],  self.mod)
        a[11] = gmpy2.mod(a[11] * self.m[2],  self.mod)
        a[13] = gmpy2.mod(a[13] * self.m[3],  self.mod)
        s[0]  = gmpy2.mod(s[0]  * self.m[4],  self.mod)
        s[1]  = gmpy2.mod(s[1]  * self.m[5],  self.mod)
        a[3]  = gmpy2.mod(a[3]  * self.m[6],  self.mod)
        a[12] = gmpy2.mod(a[12] * self.m[7],  self.mod)
        a[10] = gmpy2.mod(a[10] * self.m[8],  self.mod)
        a[14] = gmpy2.mod(a[14] * self.m[9],  self.mod)
        a[5]  = gmpy2.mod(a[5]  * self.m[10], self.mod)
        a[7]  = gmpy2.mod(a[7]  * self.m[11], self.mod)
        a[6]  = gmpy2.mod(a[6]  * self.m[12], self.mod)

        # Stage 4
        s[0], s[1]   = self.hadamard(s[0], s[1])
        a[1],  a[3]  = self.hadamard(a[1],  a[3])
        a[2],  a[6]  = self.hadamard(a[2],  a[6])

        a[10], a[14] = self.hadamard(a[10], a[14])
        a[9],  a[15] = self.hadamard(a[9],  a[15])
        a[11], a[13] = self.hadamard(a[11], a[13])
        a[5],  a[7]  = self.hadamard(a[5],  a[7])

        # Stage 5
        for i, j in [(1,5), (3,7), (9,11), (15,13)]:
            a[i], a[j] = self.hadamard(a[i], a[j])
        a[9]  = (s[0] - a[9])  % self.mod
        a[15] = (s[1] - a[15]) % self.mod

        # Output
        for i, j in [(0,8), (1,9), (2,10), (4,12), (6,14), (7,15),(3,13),(5,11)]:
            a[i], a[j] = self.hadamard(a[i], a[j])
        temp = a[13]
        a[13] = a[11]
        a[11] = temp
        return a



# --- (필요시) Radix-2 / Radix-8 BFU: 네 코드로 교체 ---
class Radix2_BFU:
    def __init__(self, mod=8380417, omega=1):
        self.mod = mod
    def process(self, a):
        assert len(a)==2
        x0 = (a[0]+a[1]) % self.mod
        x1 = (a[0]-a[1]) % self.mod
        return [x0, x1]

class Radix8_BFU:
    def __init__(self, mod=17, omega=9):
        self.mod = mod
        self.omega = omega
        # Compute inverse of 2 mod self.mod (since mod is odd prime, inv2 = (mod+1)//2)
        inv2 = (mod + 1) // 2

        # Precompute the 8 “base twiddle” values m[0]…m[7]
        m = [0] * 8
        # m[0], m[1], m[2], m[4] = 1
        for i in (0, 1, 2, 4):
            m[i] = 1
        # m[3] = m[5] = ω²
        w2 = pow(omega, 2, mod)
        m[3] = m[5] = w2
        # m[6] = (ω¹ + ω³) / 2
        w1 = pow(omega, 1, mod)
        w3 = pow(omega, 3, mod)
        m[6] = ((w1 + w3) * inv2) % mod
        # m[7] = (ω¹ − ω³) / 2
        m[7] = ((w1 - w3) * inv2) % mod

        self.m = m

    def hadamard(self, a, b):
        # (a+b) mod, (a−b) mod
        return ((a + b) % self.mod, (a - b) % self.mod)

    def process(self, a):
        # Step 1: pairwise Hadamard on the 8 inputs
        a[0], a[1] = self.hadamard(a[0], a[1])
        a[2], a[3] = self.hadamard(a[2], a[3])
        a[4], a[5] = self.hadamard(a[4], a[5])
        a[6], a[7] = self.hadamard(a[6], a[7])

        # Step 2: Hadamard on the results of Step 1 (middle stage)
        a[4],  a[6] = self.hadamard(a[4],  a[6])
        a[5],  a[7] = self.hadamard(a[5],  a[7])

        # Step 3: multiply certain lanes by their twiddle factors
        a[3] = (a[3] * self.m[3]) % self.mod
        a[6] = (a[6] * self.m[5]) % self.mod
        a[5] = (a[5] * self.m[6]) % self.mod
        a[7] = (a[7] * self.m[7]) % self.mod

        # Step 4: another round of small Hadamard butterflies
        a[0], a[2] = self.hadamard(a[0],  a[2])
        a[1], a[3] = self.hadamard(a[1],  a[3])
        a[5], a[7] = self.hadamard(a[5],  a[7])

        # Step 5: final Hadamard to produce 8 outputs
        a[0], a[4] = self.hadamard(a[0], a[4])
        a[1], a[5] = self.hadamard(a[1], a[5])
        a[2], a[6] = self.hadamard(a[2], a[6])
        a[3], a[7] = self.hadamard(a[3], a[7])

        return a

# =========================================================
# drop-1 flat twiddle 생성 (플랜 일반화)
# =========================================================
def build_flat_twiddles_drop1(
    N: int,
    stages: List[Tuple[int, type, Dict]],
    mod: int,
    gamma: int,
) -> Tuple[List[List[int]], List[Tuple[int,int]]]:
    flat_tw: List[List[int]] = []
    meta: List[Tuple[int,int]] = []
    j = 1
    for (R, _BFU, _ctor_kwargs) in stages:
        index = N // (R * j)
        gp = gmpy2.powmod(gamma, index, mod)  # gamma_pw
        rev_bits = int(math.log2(R))
        arr = [0] * (j * (R - 1))
        for j1 in range(j):
            tlist = [int(gmpy2.powmod(gp, i * (2*j1 + 1), mod)) for i in range(R)]
            tlist = [tlist[bit_reverse_int(i, rev_bits)] for i in range(R)]
            assert tlist[0] == 1, "lane-0 twiddle must be 1 (drop-1)"
            base = j1 * (R - 1)
            for i in range(1, R):
                arr[base + (i-1)] = tlist[i]
        flat_tw.append(arr)
        meta.append((R, j))
        j *= R
    return flat_tw, meta

# =========================================================
# 플랜 일반화: flat-drop1 twiddle 사용, in-place NTT
#   - BFU는 플랜에 맞춰 주입
# =========================================================
def ntt_flat_drop1_inplace_generic(poly: List[int],
                                   plan: List[int],
                                   q: int,
                                   gamma: int,
                                   bfu_ctors: Dict[int, Tuple[type, Dict]]) -> List[int]:
    N = len(poly)
    P = bit_reverse_permute(poly.copy())

    # stages: [(R, BFU_cls, kwargs), ...]
    stages = [(R, bfu_ctors[R][0], bfu_ctors[R][1]) for R in plan]

    # twiddles(사전 생성) — ★ 측정에서 제외하고 외부에서 한 번만 만들 수도 있음
    flat, meta = build_flat_twiddles_drop1(N, stages, q, gamma)

    j = 1
    for s, (R, BFU_cls, ctor_kwargs) in enumerate(stages):
        bfu   = BFU_cls(**ctor_kwargs)
        index = N // (R * j)
        flat_s = flat[s]  # 길이 = j*(R-1)
        Rm1    = R - 1
        R_meta, j_meta = meta[s]
        assert (R_meta == R and j_meta == j)

        for t in range(index * j):
            k  = t // j
            j1 = t %  j
            base  = R * k * j
            addrs = [base + j1 + i*j for i in range(R)]
            off   = j1 * Rm1

            # drop-1 pre-twiddle
            for i in range(1, R):
                P[addrs[i]] = (P[addrs[i]] * flat_s[off + (i-1)]) % q

            # BFU
            a = [P[addr] for addr in addrs]
            a = bfu.process(a)
            for i in range(R):
                P[addrs[i]] = a[i]

        j *= R

    return P

# =========================================================
# 16-4-4 (네 구현) 바인딩
# =========================================================
def ntt_16_4_4_flat_drop1_inplace(poly: List[int]) -> List[int]:
    q = 8380417
    gamma = 1753
    N = 256
    assert len(poly) == N
    plan = [16,4,4]
    bfu_ctors = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return ntt_flat_drop1_inplace_generic(poly, plan, q, gamma, bfu_ctors)

# =========================================================
# 플랜별 NTT 함수 바인딩 (여기에 네 BFU 스케줄이 그대로 들어감)
# =========================================================
def ntt_8_8_4(poly: List[int]) -> List[int]:
    q = 8380417; gamma = 1753
    plan = [8,8,4]
    bfu_ctors = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return ntt_flat_drop1_inplace_generic(poly, plan, q, gamma, bfu_ctors)

def ntt_16_16(poly: List[int]) -> List[int]:
    q = 8380417; gamma = 1753
    plan = [16,16]
    bfu_ctors = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return ntt_flat_drop1_inplace_generic(poly, plan, q, gamma, bfu_ctors)

def ntt_16_2_2_2_2(poly: List[int]) -> List[int]:
    q = 8380417; gamma = 1753
    plan = [16,2,2,2,2]
    bfu_ctors = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return ntt_flat_drop1_inplace_generic(poly, plan, q, gamma, bfu_ctors)

def ntt_16_4_2_2(poly: List[int]) -> List[int]:
    q = 8380417; gamma = 1753
    plan = [16,4,2,2]
    bfu_ctors = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return ntt_flat_drop1_inplace_generic(poly, plan, q, gamma, bfu_ctors)

def ntt_16_8_2(poly: List[int]) -> List[int]:
    q = 8380417; gamma = 1753
    plan = [16,8,2]
    bfu_ctors = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return ntt_flat_drop1_inplace_generic(poly, plan, q, gamma, bfu_ctors)

# =========================================================
# 벤치마크 (t-list 생성 시간 제외)
#   - twiddle은 미리 한 번 생성 후, 실행만 타이밍
# =========================================================
def make_stages_from_plan(plan, q, gamma):
    # BFU 타입과 파라미터 (필요시 수정)
    ctor = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    return [(R, ctor[R][0], ctor[R][1]) for R in plan]

def build_twiddles_only(N, plan, q, gamma):
    stages = make_stages_from_plan(plan, q, gamma)
    flat, meta = build_flat_twiddles_drop1(N, stages, q, gamma)
    return flat, meta

def ntt_with_prebuilt_twiddles(poly, plan, q, gamma, flat, meta):
    # flat/meta를 외부에서 넘겨받아 실행만 —> 생성시간 제외
    N = len(poly)
    P = bit_reverse_permute(poly.copy())
    ctor = {
        2: (Radix2_BFU,  {"mod": q, "omega": 1}),
        4: (Radix4_BFU,  {"mod": q, "omega": 4808194}),
        8: (Radix8_BFU,  {"mod": q, "omega": 3765607}),
        16:(Radix16_BFU, {"mod": q, "omega": 5178923}),
    }
    stages = [(R, ctor[R][0], ctor[R][1]) for R in plan]

    j = 1
    for s, (R, BFU, kwargs) in enumerate(stages):
        bfu = BFU(**kwargs)
        index = N // (R*j)
        flat_s = flat[s]
        Rm1 = R-1
        R_meta, j_meta = meta[s]
        assert (R_meta==R and j_meta==j)
        for t in range(index*j):
            k = t // j
            j1= t %  j
            base = R*k*j
            addrs = [base + j1 + i*j for i in range(R)]
            off = j1*Rm1
            for i in range(1, R):
                P[addrs[i]] = (P[addrs[i]] * flat_s[off + (i-1)]) % q
            a = [P[idx] for idx in addrs]
            a = bfu.process(a)
            for i in range(R):
                P[addrs[i]] = a[i]
        j *= R
    return P

def benchmark_plans():
    q = 8380417
    gamma = 1753
    N = 256
    rnd = random.Random(0x1234)
    poly = [rnd.randrange(q) for _ in range(N)]

    plans = {
        "16-4-4":     [16,4,4],
        "8-8-4":      [8,8,4],
        "16-16":      [16,16],
        "16-2-2-2-2": [16,2,2,2,2],
        "16-4-2-2":   [16,4,2,2],
        "16-8-2":     [16,8,2],
        "8-8-2-2":      [8, 8, 2, 2],
    }

    results = []
    for name, plan in plans.items():
        # 1) twiddle 사전생성 (시간 제외)
        t0 = time.perf_counter()
        flat, meta = build_twiddles_only(N, plan, q, gamma)
        t1 = time.perf_counter()
        tw_time_ms = (t1 - t0) * 1000.0

        # 2) 실행만 타이밍 (여러 번 평균)
        runs = 100
        elapsed = 0.0
        out_check = None
        for _ in range(runs):
            a = poly[:]  # 동일 입력
            t2 = time.perf_counter()
            out = ntt_with_prebuilt_twiddles(a, plan, q, gamma, flat, meta)
            t3 = time.perf_counter()
            elapsed += (t3 - t2)
            if out_check is None:
                out_check = out
            else:
                assert out == out_check, "Non-deterministic result?"

        avg_ms = (elapsed / runs) * 1000.0
        results.append((name, avg_ms, tw_time_ms))

    # 표 출력
    width = max(len(n) for n,_ ,_ in results) + 2
    print("\nNTT execution time (drop-1 flat, twiddle build excluded)")
    print("-"*(width+36))
    print(f"{'Plan'.ljust(width)}  {'Exec avg [ms]':>14}  {'Twiddle build [ms]*':>18}")
    print("-"*(width+36))
    for name, avg_ms, tw_ms in results:
        print(f"{name.ljust(width)}  {avg_ms:14.3f}  {tw_ms:18.3f}")
    print("-"*(width+36))
    print("* Twiddle build time is reported but not included in Exec avg.\n")

if __name__ == "__main__":
    benchmark_plans()



NTT execution time (drop-1 flat, twiddle build excluded)
------------------------------------------------
Plan           Exec avg [ms]  Twiddle build [ms]*
------------------------------------------------
16-4-4                 1.029               0.608
8-8-4                  0.727               0.381
16-16                  0.874               0.210
16-2-2-2-2             1.078               0.318
16-4-2-2               1.034               0.421
16-8-2                 0.800               0.512
8-8-2-2                0.742               0.457
------------------------------------------------
* Twiddle build time is reported but not included in Exec avg.



In [5]:
# === 아래 네 줄만 네 파일 맨 아래에 추가해서 돌리면 됩니다 ===
if __name__ == "__main__":
    q = 8380417
    gamma = 1753
    plan = [8,8,2,2]

    poly = list(range(256))  # 입력 0..255

    flat, meta = build_twiddles_only(N=256, plan=plan, q=q, gamma=gamma)
    out = ntt_with_prebuilt_twiddles(poly, plan, q, gamma, flat, meta)

    # 보기 좋게 출력
    print("NTT(0..255) with plan 16-16:")
    for i in range(0, 256, 16):
        print(out[i:i+16])


NTT(0..255) with plan 16-16:
[8023823, 8368027, 4046506, 252176, 2269315, 2703904, 7301606, 1128875, 8200405, 5432259, 7784774, 155305, 886499, 1101657, 3043243, 7276117]
[1447936, 2679432, 1077579, 6998811, 5932184, 8137948, 3069985, 1618324, 6558166, 3076903, 6404829, 915442, 7671751, 6246419, 8357109, 2425536]
[4077164, 2729052, 3351905, 4282041, 437727, 2871386, 3984450, 6308070, 8329041, 3173849, 1807242, 5498888, 503654, 4615923, 2083871, 3904031]
[7966085, 4495896, 1227026, 5078866, 5579212, 3297580, 5405335, 6421303, 490458, 4888110, 167925, 8363923, 1880030, 1264622, 3142318, 6555455]
[5503697, 144560, 861568, 3998553, 65279, 1178408, 6224608, 4152979, 5840697, 3304060, 2489952, 5776192, 1715375, 2667995, 3966814, 1301221]
[3833152, 3256914, 1063576, 528380, 4400120, 2930707, 5559452, 13008, 592160, 2874775, 699854, 1281704, 6190567, 3436132, 3444419, 3777265]
[2287113, 1224642, 8240568, 1284601, 45209, 4524768, 7183502, 3563602, 7624292, 8209741, 4210121, 6047002, 3475364, 78

In [7]:
if __name__ == "__main__":
    q = 8380417
    gamma = 1753
    plan = [8,4,4,2]

    poly = list(range(256))  # 입력 0..255

    flat, meta = build_twiddles_only(N=256, plan=plan, q=q, gamma=gamma)
    out = ntt_with_prebuilt_twiddles(poly, plan, q, gamma, flat, meta)

    # 보기 좋게 출력
    print("NTT(0..255) with plan 16-16:")
    for i in range(0, 256, 16):
        print(out[i:i+16])


NTT(0..255) with plan 16-16:
[8023823, 8368027, 4046506, 252176, 2269315, 2703904, 7301606, 1128875, 8200405, 5432259, 7784774, 155305, 886499, 1101657, 3043243, 7276117]
[1447936, 2679432, 1077579, 6998811, 5932184, 8137948, 3069985, 1618324, 6558166, 3076903, 6404829, 915442, 7671751, 6246419, 8357109, 2425536]
[4077164, 2729052, 3351905, 4282041, 437727, 2871386, 3984450, 6308070, 8329041, 3173849, 1807242, 5498888, 503654, 4615923, 2083871, 3904031]
[7966085, 4495896, 1227026, 5078866, 5579212, 3297580, 5405335, 6421303, 490458, 4888110, 167925, 8363923, 1880030, 1264622, 3142318, 6555455]
[5503697, 144560, 861568, 3998553, 65279, 1178408, 6224608, 4152979, 5840697, 3304060, 2489952, 5776192, 1715375, 2667995, 3966814, 1301221]
[3833152, 3256914, 1063576, 528380, 4400120, 2930707, 5559452, 13008, 592160, 2874775, 699854, 1281704, 6190567, 3436132, 3444419, 3777265]
[2287113, 1224642, 8240568, 1284601, 45209, 4524768, 7183502, 3563602, 7624292, 8209741, 4210121, 6047002, 3475364, 78